In [1]:
import pandas as pd
import numpy as np

In [2]:
pn_units_path = "/projects/ml/RF2_allatom/datasets/pdb/2024_09_06/pn_units_df.parquet"
interfaces_path = "/projects/ml/RF2_allatom/datasets/pdb/2024_09_06/interfaces_df.parquet"

pn_unit_df = pd.read_parquet(pn_units_path)
interfaces_df = pd.read_parquet(interfaces_path)

In [11]:
# Get the AF-3 excluded ligands
AF3_EXCLUDED_LIGANDS_REGEX = "(?:^|,)\\s*(?:144|15P|1PE|2F2|2JC|3HR|3SY|7N5|7PE|9JE|AAE|ABA|ACE|ACN|ACT|ACY|AZI|BAM|BCN|BCT|BDN|BEN|BME|BO3|BTB|BTC|BU1|C8E|CAD|CAQ|CBM|CCN|CIT|CL|CLR|CM|CMO|CO3|CPT|CXS|D10|DEP|DIO|DMS|DN|DOD|DOX|EDO|EEE|EGL|EOH|EOX|EPE|ETF|FCY|FJO|FLC|FMT|FW5|GOL|GSH|GTT|GYF|HED|IHP|IHS|IMD|IOD|IPA|IPH|LDA|MB3|MEG|MES|MLA|MLI|MOH|MPD|MRD|MSE|MYR|N|NA|NH2|NH4|NHE|NO3|O4B|OHE|OLA|OLC|OMB|OME|OXA|P6G|PE3|PE4|PEG|PEO|PEP|PG0|PG4|PGE|PGR|PLM|PO4|POL|POP|PVO|SAR|SCN|SEO|SIN|SO4|SPD|SPM|SR|STE|STO|STU|TAR|TBU|TME|TRS|UNK|UNL|UNX|UPL|URE)\\s*(?:,|$)"

# For all string columns, set all NaN values to empty string
string_columns = pn_unit_df.select_dtypes(include=["object"]).columns
print("Filling NaN values in the string columns of the pn_units_df...")
pn_unit_df[string_columns] = pn_unit_df[string_columns].fillna("")

string_columns = interfaces_df.select_dtypes(include=["object"]).columns
print("Filling NaN values in the string columns of the interfaces...")
interfaces_df[string_columns] = interfaces_df[string_columns].fillna("")

# Create a new dataframe applies
print("Applying the regex to the non-polymer res names for the pn_units_df...")
excluded_ligand_pn_unit_mask = pn_unit_df["q_pn_unit_non_polymer_res_names"].str.contains(AF3_EXCLUDED_LIGANDS_REGEX)
print("Applying the regex to the non-polymer res names for the intefaces...")
excluded_ligand_interface_mask = interfaces_df["pn_unit_1_non_polymer_res_names"].str.contains(
    AF3_EXCLUDED_LIGANDS_REGEX
) | interfaces_df["pn_unit_2_non_polymer_res_names"].str.contains(AF3_EXCLUDED_LIGANDS_REGEX)

# Remove the excluded ligands
print("Filtering out the excluded ligands from the pn_units_df and the interfaces...")
pn_unit_df_filtered = pn_unit_df[~excluded_ligand_pn_unit_mask]
interfaces_df_filtered = interfaces_df[~excluded_ligand_interface_mask]

Filling NaN values in the string columns of the pn_units_df...
Applying the regex to the non-polymer res names for the pn_units_df...
Applying the regex to the non-polymer res names for the intefaces...
Filtering out the excluded ligands from the pn_units_df and the interfaces...


In [13]:
print(len(pn_unit_df_filtered), len(interfaces_df_filtered))
print(len(pn_unit_df), len(interfaces_df))

3237065 5792229
3452644 6165001


In [14]:
# Save the filtered dataframes
print("Saving the filtered dataframes...")
pn_unit_df_filtered.to_parquet(pn_units_path.replace(".parquet", "_no_af3_ligands.parquet"))
interfaces_df_filtered.to_parquet(interfaces_path.replace(".parquet", "_no_af3_ligands.parquet"))

Saving the filtered dataframes...
